In [42]:
import os
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['font.size'] = 14
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [43]:
root_dir = '/data/Blob_EastUS/xiaofan/battery/all_temp_new/data/NMC'

In [44]:
train_list = pd.read_csv(os.path.join(root_dir, 'training.csv'))
test_in_list = pd.read_csv(os.path.join(root_dir, 'test_in.csv'))
test_out_list = pd.read_csv(os.path.join(root_dir, 'test_out.csv'))

In [45]:
import numpy as np
try:
    from scipy.integrate import cumulative_trapezoid as cumtrapz
except ImportError:
    # Fallback for older SciPy versions
    from scipy.integrate import cumtrapz

def calculate_energy(Q, V, I, t, unit='J'):
    """
    Calculate energy (W) from battery cycle data.
    
    Parameters:
        Q (array-like): charge data (Ah or C)
        V (array-like): voltage data (V)
        I (array-like): current data (A)
        t (array-like): time data (s)
        unit (str): 'J' for Joules or 'Wh' for Watt-hours
        
    Returns:
        float: total energy in the chosen unit
    """
    # Convert inputs to numpy arrays
    V = np.array(V)
    I = np.array(I)
    t = np.array(t)
    
    # Compute instantaneous power
    P = V * I  # Watts
    
    # # Integrate power over time (trapezoidal rule)
    # W_joules = np.trapz(P, t)  # Joules (W·s)
     # Cumulative energy (integrate using trapezoidal rule)
    W_joules = np.concatenate(([0], cumtrapz(P, t)))  # Joules # Joules (W·s)
    
    if unit == 'Wh':
        W = W_joules / 3600  # convert to Watt-hours
    else:
        W = W_joules
    
    return W


In [46]:
### Define a function to load RPT data and convert the data type ###
import json
def convert_RPT_to_dict(RPT_json_dir,cell,subfolder):
    with open(RPT_json_dir+subfolder+cell+'.json','r') as file:
        data_dict = json.loads(json.load(file))
    
    # Convert time series data from string to np.datetime64
    for iii, start_time in enumerate(data_dict['start_stop_time']['start']):
        if start_time != '[]':
            data_dict['start_stop_time']['start'][iii] = np.datetime64(start_time)
            data_dict['start_stop_time']['stop'][iii] = np.datetime64(data_dict['start_stop_time']['stop'][iii])
        else:
            data_dict['start_stop_time']['start'][iii] = []
            data_dict['start_stop_time']['stop'][iii] = []

    for iii in range(len(data_dict['start_stop_time']['start'])):
        data_dict['QV_charge_C_2']['t'][iii] = list(map(np.datetime64,data_dict['QV_charge_C_2']['t'][iii]))
        data_dict['QV_discharge_C_2']['t'][iii] = list(map(np.datetime64,data_dict['QV_discharge_C_2']['t'][iii]))
        data_dict['QV_charge_C_5']['t'][iii] = list(map(np.datetime64,data_dict['QV_charge_C_5']['t'][iii]))
        data_dict['QV_discharge_C_5']['t'][iii] = list(map(np.datetime64,data_dict['QV_discharge_C_5']['t'][iii]))
    
    # Return the preprocessed Python Dictionary
    return data_dict


In [47]:
# import json
# import os

batch2 = ['G57C1','G57C2','G57C3','G57C4','G58C1', 'G26C3','G49C1','G49C2','G49C3','G49C4','G50C1','G50C3','G50C4'] 

# # dir_path = '/data/Blob_EastUS/xiaofan/battery/all_temp_new/data/NMC'

valid_cells = pd.read_csv(os.path.join(root_dir, 'Valid_cells.csv')).values.flatten().tolist()
# sohs = {}
# # root_dir = '/data/Blob_EastUS/xiaofan/battery/all_temp_new/data/NMC/Cycling_json/Cycling_json/Release 2.0'
# for cell_name in valid_cells:
# # for cell_name in ['G20C1']:
#     print(cell_name)
#     # print(i)
#     group = ''
#     # cell_name = i.split('.')[0]
#     if cell_name in train_list['Cell'].values:
#         # print(cell_name)
#         group = 'train'
#     elif cell_name in test_in_list['Cell'].values:
#         group = 'test_in'
#     elif cell_name in test_out_list['Cell'].values:
#         group = 'test_out'
#     else:
#         group = 'others'

#     if cell_name in batch2:
#         subfolder = 'Release 2.0/'
#     else:
#         subfolder = 'Release 1.0/'
#     # if os.path.exists(f'/data/Blob_EastUS/xiaofan/battery/all_temp_new/data/NMC/processed/{group}/{cell_name}.pkl'):
#     #     print(f'{cell_name} already existed, skip...')
#     #     continue

#     file_path = f'{root_dir}/Cycling_json/Cycling_json/{subfolder}/{cell_name}.json'

#     with open(file_path,'r') as file:
#         data_dict = json.loads(json.load(file))
#     RPT_json_dir = os.path.join(root_dir, 'RPT_json/')
#     RPT_dict = convert_RPT_to_dict(RPT_json_dir,cell_name, subfolder)
#     # print(RPT_dict['start_stop_time']['start'])
#     new_data = []
#     info = data_dict['QV_charge']
#     discharge_time_each_cycle = []
#     for cycle_idx in range(len(data_dict['capacity_discharge'])):
#         flag = False
#         cycle_info = {}
#         for mode in ['c', 'd']:
#             for element in ['Q', 'V',  'E', 'I']:
#                 # print(info['Q'][cycle_idx])
#                 cycle_info[f'{element}_{mode}'] = info[element][cycle_idx]
#             # W
#             try:
#                 if mode =='d':
#                     discharge_time_each_cycle.append(info['t'][cycle_idx][0])
#                 times = np.array(info['t'][cycle_idx], dtype='datetime64[ns]')
#                 t_seconds = ( times- times[0]) / np.timedelta64(1, 's')
#                 cycle_info[f't_{mode}'] = t_seconds

#                 cycle_info[f'W_{mode}']  = calculate_energy(info['Q'][cycle_idx], info['V'][cycle_idx], info['I'][cycle_idx], t_seconds, unit='J')
                
#             except:
#                 flag = True
#                 print(f'{cell_name} {cycle_idx} error')
#         if flag == False:
#             new_data.append(cycle_info)
    
#     discharge_time_each_cycle = np.array(discharge_time_each_cycle, dtype='datetime64[ns]')
#     cleaned = [
#         np.datetime64('NaT') if isinstance(x, list) and len(x) == 0 else x
#         for x in RPT_dict['start_stop_time']['start']
#     ]
    
#     rpt_time = np.array(cleaned, dtype='datetime64[ns]')
    
#     indices = np.searchsorted(discharge_time_each_cycle, rpt_time)
#     # print(indices)
#     sohs[cell_name] = {}
#     for rpt_idx, i in enumerate(indices):
#         # print(rpt_idx, i, RPT_dict['capacity_discharge_C_5'][rpt_idx])
#         sohs[cell_name][int(i)] = RPT_dict['capacity_discharge_C_5'][rpt_idx]
#     # except:
#     #     print(rpt_time)

#     # import pickle
#     # # 将列表保存到 pickle 文件
#     # with open(f'/data/Blob_EastUS/xiaofan/battery/all_temp_new/data/NMC/processed/{group}/{cell_name}.pkl', 'wb') as file:
#     #     pickle.dump(new_data, file)
#     # break
# import json

# with open("sohs.json", "w") as f:
#     json.dump(sohs, f, indent=4)

# import pickle

# with open("sohs.pkl", "wb") as f:
#     pickle.dump(sohs, f)


In [48]:
import numpy as np

def cycle_for_soh(soh_dict, target=None):
    # sort by cycle index
    if not target:
        target = soh_dict[0] * 0.8
    
    cleaned = {k: v for k, v in soh_dict.items() if isinstance(v, (int, float, np.floating))}
    items = sorted(cleaned.items())
    cycles = np.array([c for c, _ in items], dtype=float)
    sohs = np.array([s for _, s in items], dtype=float)

    # exact match first
    eq_idx = np.where(np.isclose(sohs, target))[0]
    if eq_idx.size:
        return float(cycles[eq_idx[0]])

    # find a segment that brackets the target
    for i in range(len(sohs) - 1):
        s0, s1 = sohs[i], sohs[i + 1]
        if s0 == s1:
            continue
        if (s0 - target) * (s1 - target) < 0 or np.isclose([s0], target) or np.isclose([s1], target):
            c0, c1 = cycles[i], cycles[i + 1]
            # linear interpolation
            t = (target - s0) / (s1 - s0)
            return float(c0 + t * (c1 - c0))

    # If not bracketed, you can choose to return None or extrapolate
    return None  # or do extrapolation if desired


In [49]:
# rul_dict = {}
# for cell_name, soh in sohs.items():
#     # print(cell_name)
#     rul_dict[cell_name] = int(cycle_for_soh(soh))


# import json

# with open("/data/Blob_EastUS/xiaofan/battery/all_temp_new/data/NMC/rul_dict.json", "w") as f:
#     json.dump(rul_dict, f, indent=4)


# Read data and extract feature

In [50]:
from src.io import load_data
# data_folder = Path.cwd().parent / 'data/raw'

data_folder = os.path.join(root_dir, 'processed/train')
train_data = load_data(data_folder,  is_ours=False, interp_dims=1000, Q_min=0, Q_max=0.3, V_min=3.1, V_max=4.65)



Loading cells: 100%|██████████| 116/116 [38:25<00:00, 19.87s/it]  


In [51]:
data_folder = os.path.join(root_dir, 'processed/test_in')
test_in_data = load_data(data_folder,  is_ours=False, interp_dims=1000, Q_min=0, Q_max=0.3, V_min=3.1, V_max=4.65)

data_folder = os.path.join(root_dir, 'processed/test_out')
test_out_data = load_data(data_folder,  is_ours=False, interp_dims=1000, Q_min=0, Q_max=0.3, V_min=3.1, V_max=4.65)
# fixed_labels = annotate_labels(data, label_mode='fixed')
# relative_labels = annotate_labels(data, label_mode='relative')

Loading cells:   3%|▎         | 2/60 [00:17<08:44,  9.04s/it]

Loading cells: 100%|██████████| 49/49 [40:27<00:00, 49.53s/it]   


# Load feature

In [52]:
# /data/Blob_EastUS/xiaofan/battery/all_temp_new/data/NMC/
# capacity_dir = os.path.join(root_dir, 'capacity_fade')

label_df = pd.read_csv(os.path.join(root_dir, 'feature_all.csv'))
label_dict = dict(zip(label_df['Cell'], label_df['Lifetime']))


In [53]:
# for cell_name in valid_cells:
#     print(cell_name)
#     if cell_name in batch2:
#         subfolder = 'Release 2.0/'
#     else:
#         subfolder = 'Release 1.0/'

#     capacity_path = os.path.join(capacity_dir, subfolder, f'{cell_name}.csv')
#     capacity = 

In [54]:
# train_labels = {k: rul_dict[k] for k, v in train_data.items()}
# test_in_labels = {k: rul_dict[k] for k, v in test_in_data.items()}
# test_out_labels = {k: rul_dict[k] for k, v in test_out_data.items()}

In [55]:
# concat train, test_in and test_out labels
merged_data = train_data | test_in_data | test_out_data
# merged_ruls = train_labels | test_in_labels | test_out_labels
merged_ruls = label_dict


In [63]:
import torch 
num_early_cycles = 0

X = torch.stack([
    torch.stack([
        torch.stack([
            cycle_data['VQ_c'], cycle_data['VQ_d'],
            cycle_data['dVdQ_c'], cycle_data['dVdQ_d'], 
            cycle_data['interp_I_c'], cycle_data['interp_I_d'],
            cycle_data['interp_V_c'], cycle_data['interp_V_d'],
            cycle_data['QV_c'], cycle_data['QV_d'],
            cycle_data['interp_E_c'], cycle_data['interp_E_d'],
            cycle_data['interp_W_c'], cycle_data['interp_W_d']
        ]) for cycle_data in merged_data[cell][:50]
    ]) for cell, label in merged_ruls.items() if label > num_early_cycles
]).permute(0, 2, 1, 3).contiguous()

y = np.array([
    label for _, label in merged_ruls.items() if label > num_early_cycles
])

select_tags= [cell for cell, label in merged_ruls.items() if label > num_early_cycles]

In [60]:
train_indices = [select_tags.index(tag) for tag in train_list['Cell'].values if tag in select_tags]
# print(train_indices)

test_in_indices = [select_tags.index(tag) for tag in test_in_list['Cell'].values if tag in select_tags]
# print(test_in_indices)

test_out_indices = [select_tags.index(tag) for tag in test_out_list['Cell'].values if tag in select_tags]
# print(test_out_indices)


In [44]:
len(test_out_indices)

49

In [39]:
len(test_in_indices)

60

In [57]:
def print_score(model_score):
    mean_score = np.mean(model_score, axis=0).squeeze(), np.std(model_score, axis=0).squeeze()
    train_RMSE, test_RMSE, train_RMSE_variance, test_RMSE_variance =  mean_score[0][0][0], mean_score[0][0][1], mean_score[1][0][0], mean_score[1][0][1]
    train_MAPE, test_MAPE, train_MAPE_variance, test_MAPE_variance  =  mean_score[0][1][0], mean_score[0][1][1], mean_score[1][1][0], mean_score[1][1][1]
    print(f'train_RMSE:{train_RMSE:.2f} ± {train_RMSE_variance:.0f}, test_RMSE:{test_RMSE:.2f} ± {test_RMSE_variance:.0f}' )
    print(f'train_MAPE:{train_MAPE:.2f} ± {train_MAPE_variance:.0f}, test_MAPE:{test_MAPE:.2f} ± {test_MAPE_variance:.0f}' )

In [64]:
from src.learning import random_fit_and_eval, RMSE, MAPE
from src.utils import (
    diff_cycles,
    savgol_smooth,
    nankurtosis,
    nanmax,
    nanmin,
    nanmean,
    nanskew,
    nanvar
)
from src.feature import our_feature


X_ours, fea_names = our_feature(
    X,
    num_cycle_groups=5,
    num_signal_segments=4,
    aggregators=[
        nankurtosis, nanmax, nanmin, nanmean, nanvar, nanskew
    ],
    activators=[
        lambda x: x,  # identity
        torch.abs
    ],
    build_feature_name= True,
    device='cuda:0'
)
print(X_ours.shape, len(fea_names))
X_ours = np.nan_to_num(X_ours, nan=0.0, posinf=0.0, neginf=0.0)




(225, 60480) 60480


In [53]:

from tqdm import tqdm
from sklearn.ensemble import RandomForestRegressor

key = 'Ours_test_in'

# scores = {}
seeds = list(range(16))  # Randomness for dataset split and model initialization
evaluators = [RMSE(log_scale=True), MAPE(log_scale=True)]
num_early_cycles = 50

# smoother = partial(savgol_smooth, window=51, order=3)

scores = {}
predictions = {}
scores[key] = []
predictions[key] = []
# feature_importances = []
# TODO: change the function to input train and test
for seed in tqdm(seeds, desc=f'Processing {key}'):
    score, test_y, test_pred = random_fit_and_eval(
        X_ours, np.log(y), [RandomForestRegressor()], evaluators, seed=seed, return_prediction= True, return_train_loss=True, train_idx=train_indices, test_idx=test_in_indices
    )
    # score, feature_importance= random_fit_and_eval(
    #     X_ours, np.log(y), [RandomForestRegressor()], evaluators, seed=seed, feature_name= fea_names, return_train_loss=True
    # )
    scores[key].append(score)
    predictions[key].append(test_pred)

print(f'model: {key}')
print_score(scores[key])

Processing Ours_test_in:   0%|          | 0/16 [00:00<?, ?it/s]

Processing Ours_test_in: 100%|██████████| 16/16 [3:31:29<00:00, 793.10s/it]  

model: Ours_test_in
train_RMSE:1.60 ± 0, test_RMSE:3.67 ± 0
train_MAPE:6.96 ± 0, test_MAPE:21.60 ± 1


In [54]:

from tqdm import tqdm
from sklearn.ensemble import RandomForestRegressor

key = 'Ours_test_out'

scores[key] = []
predictions[key] = []
# feature_importances = []
# TODO: change the function to input train and test
for seed in tqdm(seeds, desc=f'Processing {key}'):
    score, test_y, test_pred = random_fit_and_eval(
        X_ours, np.log(y), [RandomForestRegressor()], evaluators, seed=seed, return_prediction= True, return_train_loss=True, train_idx=train_indices, test_idx=test_out_indices
    )
    # score, feature_importance= random_fit_and_eval(
    #     X_ours, np.log(y), [RandomForestRegressor()], evaluators, seed=seed, feature_name= fea_names, return_train_loss=True
    # )
    scores[key].append(score)
    predictions[key].append(test_pred)

print(f'model: {key}')
print_score(scores[key])

Processing Ours_test_out: 100%|██████████| 16/16 [3:33:30<00:00, 800.64s/it]  

model: Ours_test_out
train_RMSE:1.60 ± 0, test_RMSE:14.36 ± 0
train_MAPE:6.96 ± 0, test_MAPE:30.33 ± 1


In [ ]:
# 50 圈：cycle group 5
model: Ours_test_in
train_RMSE:1.63 ± 0, test_RMSE:3.94 ± 0
train_MAPE:7.93 ± 0, test_MAPE:26.17 ± 0
model: Ours_test_out
train_RMSE:1.63 ± 0, test_RMSE:14.32 ± 0
train_MAPE:7.93 ± 0, test_MAPE:31.09 ± 1

# 100 圈：cycle group 5
model: Ours_test_in
train_RMSE:1.64 ± 0, test_RMSE:3.88 ± 0
train_MAPE:7.90 ± 0, test_MAPE:24.64 ± 1

model: Ours_test_out
train_RMSE:1.64 ± 0, test_RMSE:14.57 ± 0
train_MAPE:7.90 ± 0, test_MAPE:31.68 ± 1

# 300圈 cycle group 10

model: Ours_test_in
train_RMSE:1.60 ± 0, test_RMSE:3.67 ± 0
train_MAPE:6.96 ± 0, test_MAPE:21.60 ± 1

model: Ours_test_out
train_RMSE:1.60 ± 0, test_RMSE:14.36 ± 0
train_MAPE:6.96 ± 0, test_MAPE:30.33 ± 1

In [ ]:
# 用RPT的信息作为特征输入
# 前3周


In [10]:

for cell_name in valid_cells:
    print(cell_name)

    if cell_name in batch2:
        subfolder = 'Release 2.0/'
    else:
        subfolder = 'Release 1.0/'

    # file_path = f'{root_dir}/Cycling_json/Cycling_json/{subfolder}/{cell_name}.json'

    # with open(file_path,'r') as file:
    #     data_dict = json.loads(json.load(file))
    RPT_json_dir = os.path.join(root_dir, 'RPT_json/')
    RPT_dict = convert_RPT_to_dict(RPT_json_dir,cell_name, subfolder)
    # print(RPT_dict['start_stop_time']['start'])
    break

G1C1


In [11]:
RPT_dict.keys()

dict_keys(['capacity_discharge_C_5', 'capacity_discharge_C_2', 'capacity_charge_C_5', 'capacity_charge_C_2', 'QV_discharge_C_5', 'QV_charge_C_5', 'QV_discharge_C_2', 'QV_charge_C_2', 'start_stop_time'])

In [14]:
RPT_dict['capacity_discharge_C_5'][0]

0.28045655555555554

In [17]:
RPT_dict['QV_discharge_C_5'].keys()

dict_keys(['Q', 'V', 't', 'E', 'I'])

In [29]:
sohs = {}
# root_dir = '/data/Blob_EastUS/xiaofan/battery/all_temp_new/data/NMC/Cycling_json/Cycling_json/Release 2.0'
for cell_name in valid_cells:
# for cell_name in ['G20C1']:
    print(cell_name)

    if cell_name in batch2:
        subfolder = 'Release 2.0/'
    else:
        subfolder = 'Release 1.0/'

    RPT_json_dir = os.path.join(root_dir, 'RPT_json/')
    RPT_dict = convert_RPT_to_dict(RPT_json_dir,cell_name, subfolder)
    # print(RPT_dict['start_stop_time']['start'])
    new_data = []

    

    for cycle_idx in range(len(RPT_dict['capacity_discharge_C_5'])):
        flag = False
        cycle_info = {}
        for mode in ['c', 'd']:
            if mode =='d':
                info = RPT_dict['QV_discharge_C_5']
            else:
                info = RPT_dict['QV_charge_C_5']

            for element in ['Q', 'V',  'E', 'I']:
                # print(info['Q'][cycle_idx])
                cycle_info[f'{element}_{mode}'] = info[element][cycle_idx]
            # W
            try:

                times = np.array(info['t'][cycle_idx], dtype='datetime64[ns]')
                t_seconds = ( times- times[0]) / np.timedelta64(1, 's')
                cycle_info[f't_{mode}'] = t_seconds

                cycle_info[f'W_{mode}']  = calculate_energy(info['Q'][cycle_idx], info['V'][cycle_idx], info['I'][cycle_idx], t_seconds, unit='J')
                
            except:
                flag = True
                print(f'{cell_name} {cycle_idx} error')
        if flag == False:
            new_data.append(cycle_info)
    

    import pickle
    # 将列表保存到 pickle 文件
    with open(os.path.join(root_dir, 'RPT_processed', f'{cell_name}.pkl'), 'wb') as file:
        pickle.dump(new_data, file)
    # break


G1C1
G1C2
G1C3
G1C4
G2C1
G2C2
G2C3
G2C4
G3C1
G3C2
G3C3
G3C4
G4C1
G4C2
G4C3
G4C4
G5C1
G5C2
G5C3
G5C4
G6C1
G6C2
G6C3
G6C4
G7C1
G7C2
G7C3
G7C4
G8C1
G8C2
G8C3
G8C4
G9C1
G9C2
G9C3
G9C4
G10C1
G10C2
G10C3
G10C4
G11C1
G11C2
G11C3
G11C4
G12C1
G12C2
G12C3
G12C4
G13C1
G13C2
G13C3
G13C4
G14C1
G14C2
G14C3
G14C4
G16C1
G16C2
G16C3
G16C4
G17C1
G17C2
G17C3
G17C4
G18C1
G18C2
G18C3
G18C4
G19C1
G19C2
G19C3
G19C4
G20C1
G20C1 5 error
G20C1 5 error
G20C2
G20C2 5 error
G20C2 5 error
G20C3
G20C3 5 error
G20C3 5 error
G20C4
G20C4 5 error
G20C4 5 error
G21C1
G21C1 5 error
G21C1 5 error
G21C2
G21C2 5 error
G21C2 5 error
G21C3
G21C3 5 error
G21C3 5 error
G21C4
G21C4 5 error
G21C4 5 error
G22C1
G22C1 5 error
G22C1 5 error
G22C2
G22C2 5 error
G22C2 5 error
G22C3
G22C3 5 error
G22C3 5 error
G22C4
G22C4 5 error
G22C4 5 error
G23C1
G23C1 5 error
G23C1 5 error
G23C2
G23C2 5 error
G23C2 5 error
G23C3
G23C3 5 error
G23C3 5 error
G23C4
G23C4 5 error
G23C4 5 error
G24C1
G24C1 5 error
G24C1 5 error
G24C2
G24C2 5 error
G24C2 

In [34]:
from src.io import load_data
# data_folder = Path.cwd().parent / 'data/raw'

data_folder = os.path.join(root_dir, 'RPT_processed')
# QV range need to adjust
merged_data = load_data(data_folder,  is_ours=False, interp_dims=1000, Q_min=0, Q_max=0.3, V_min=2.999, V_max=4.21)



Loading cells: 100%|██████████| 251/251 [02:10<00:00,  1.92it/s]


In [35]:
label_df = pd.read_csv(os.path.join(root_dir, 'feature_all.csv'))
merged_ruls = dict(zip(label_df['Cell'], label_df['Lifetime']))

In [36]:
import torch 
num_early_cycles = 4

X = torch.stack([
    torch.stack([
        torch.stack([
            cycle_data['VQ_c'], cycle_data['VQ_d'],
            cycle_data['dVdQ_c'], cycle_data['dVdQ_d'], 
            cycle_data['interp_I_c'], cycle_data['interp_I_d'],
            cycle_data['interp_V_c'], cycle_data['interp_V_d'],
            cycle_data['QV_c'], cycle_data['QV_d'],
            cycle_data['interp_E_c'], cycle_data['interp_E_d'],
            cycle_data['interp_W_c'], cycle_data['interp_W_d']
        ]) for cycle_data in merged_data[cell][:4]
    ]) for cell, label in merged_ruls.items() if label > num_early_cycles
]).permute(0, 2, 1, 3).contiguous()

y = np.array([
    label for _, label in merged_ruls.items() if label > num_early_cycles
])

select_tags= [cell for cell, label in merged_ruls.items() if label > num_early_cycles]

In [37]:
train_indices = [select_tags.index(tag) for tag in train_list['Cell'].values if tag in select_tags]
# print(train_indices)

test_in_indices = [select_tags.index(tag) for tag in test_in_list['Cell'].values if tag in select_tags]
# print(test_in_indices)

test_out_indices = [select_tags.index(tag) for tag in test_out_list['Cell'].values if tag in select_tags]
# print(test_out_indices)


In [39]:
def print_score(model_score):
    mean_score = np.mean(model_score, axis=0).squeeze(), np.std(model_score, axis=0).squeeze()
    train_RMSE, test_RMSE, train_RMSE_variance, test_RMSE_variance =  mean_score[0][0][0], mean_score[0][0][1], mean_score[1][0][0], mean_score[1][0][1]
    train_MAPE, test_MAPE, train_MAPE_variance, test_MAPE_variance  =  mean_score[0][1][0], mean_score[0][1][1], mean_score[1][1][0], mean_score[1][1][1]
    print(f'train_RMSE:{train_RMSE:.2f} ± {train_RMSE_variance:.0f}, test_RMSE:{test_RMSE:.2f} ± {test_RMSE_variance:.0f}' )
    print(f'train_MAPE:{train_MAPE:.2f} ± {train_MAPE_variance:.0f}, test_MAPE:{test_MAPE:.2f} ± {test_MAPE_variance:.0f}' )


from src.learning import random_fit_and_eval, RMSE, MAPE
from src.utils import (
    diff_cycles,
    savgol_smooth,
    nankurtosis,
    nanmax,
    nanmin,
    nanmean,
    nanskew,
    nanvar
)
from src.feature import our_feature


X_ours, fea_names = our_feature(
    X,
    num_cycle_groups=4,
    num_signal_segments=4,
    aggregators=[
        nankurtosis, nanmax, nanmin, nanmean, nanvar, nanskew
    ],
    activators=[
        lambda x: x,  # identity
        torch.abs
    ],
    build_feature_name= True,
    device='cuda:0'
)
print(X_ours.shape, len(fea_names))
X_ours = np.nan_to_num(X_ours, nan=0.0, posinf=0.0, neginf=0.0)




(224, 40320) 40320


In [40]:

from tqdm import tqdm
from sklearn.ensemble import RandomForestRegressor

key = 'Ours_test_in'

# scores = {}
seeds = list(range(16))  # Randomness for dataset split and model initialization
evaluators = [RMSE(log_scale=True), MAPE(log_scale=True)]
num_early_cycles = 50

# smoother = partial(savgol_smooth, window=51, order=3)

scores = {}
predictions = {}
scores[key] = []
predictions[key] = []
# feature_importances = []
# TODO: change the function to input train and test
for seed in tqdm(seeds, desc=f'Processing {key}'):
    score, test_y, test_pred = random_fit_and_eval(
        X_ours, np.log(y), [RandomForestRegressor()], evaluators, seed=seed, return_prediction= True, return_train_loss=True, train_idx=train_indices, test_idx=test_in_indices
    )
    # score, feature_importance= random_fit_and_eval(
    #     X_ours, np.log(y), [RandomForestRegressor()], evaluators, seed=seed, feature_name= fea_names, return_train_loss=True
    # )
    scores[key].append(score)
    predictions[key].append(test_pred)

print(f'model: {key}')
print_score(scores[key])



Processing Ours_test_in: 100%|██████████| 16/16 [19:49<00:00, 74.37s/it]

model: Ours_test_in
train_RMSE:1.35 ± 0, test_RMSE:3.09 ± 0
train_MAPE:6.45 ± 0, test_MAPE:16.10 ± 1


In [41]:

from tqdm import tqdm
from sklearn.ensemble import RandomForestRegressor

key = 'Ours_test_out'

# scores = {}
seeds = list(range(16))  # Randomness for dataset split and model initialization
evaluators = [RMSE(log_scale=True), MAPE(log_scale=True)]
num_early_cycles = 50

# smoother = partial(savgol_smooth, window=51, order=3)

scores = {}
predictions = {}
scores[key] = []
predictions[key] = []
# feature_importances = []
# TODO: change the function to input train and test
for seed in tqdm(seeds, desc=f'Processing {key}'):
    score, test_y, test_pred = random_fit_and_eval(
        X_ours, np.log(y), [RandomForestRegressor()], evaluators, seed=seed, return_prediction= True, return_train_loss=True, train_idx=train_indices, test_idx=test_out_indices
    )
    # score, feature_importance= random_fit_and_eval(
    #     X_ours, np.log(y), [RandomForestRegressor()], evaluators, seed=seed, feature_name= fea_names, return_train_loss=True
    # )
    scores[key].append(score)
    predictions[key].append(test_pred)

print(f'model: {key}')
print_score(scores[key])



Processing Ours_test_out: 100%|██████████| 16/16 [20:06<00:00, 75.43s/it]

model: Ours_test_out
train_RMSE:1.35 ± 0, test_RMSE:12.62 ± 0
train_MAPE:6.45 ± 0, test_MAPE:23.63 ± 1


In [ ]:
model: Ours_test_in
train_RMSE:1.35 ± 0, test_RMSE:3.09 ± 0
train_MAPE:6.45 ± 0, test_MAPE:16.10 ± 1


model: Ours_test_out
train_RMSE:1.35 ± 0, test_RMSE:12.62 ± 0
train_MAPE:6.45 ± 0, test_MAPE:23.63 ± 1